In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json

In [2]:
import plotly.express as px
import pandas as pd

# ── 1. Load & filter ──────────────────────────────────────────────────────────
gm = px.data.gapminder()
gm_2007 = gm[gm["year"] == 2007].copy()

# ── 2. Compute deviation from global mean ────────────────────────────────────
global_mean = gm_2007["lifeExp"].mean()          # ≈ 67.0 years
gm_2007["dev"] = gm_2007["lifeExp"] - global_mean

# Round for cleaner hover labels
gm_2007["dev_r"]     = gm_2007["dev"].round(1)
gm_2007["lifeExp_r"] = gm_2007["lifeExp"].round(1)
gm_2007["mean_r"]    = round(global_mean, 1)

# ── 3. Build the map ──────────────────────────────────────────────────────────


fig = px.choropleth(
    gm_2007,
    locations        = "iso_alpha",          # ISO-3 codes in the Gapminder df
    locationmode     = "ISO-3",              # matches Plotly's built-in GeoJSON
    color            = "dev",               # diverging variable
    color_continuous_scale = "RdBu",        # diverging: red = below, blue = above
    color_continuous_midpoint = 0,          # white anchored at world average
    hover_name       = "country",
    hover_data       = {
        "lifeExp_r": True,   # raw life expectancy (rounded)
        "dev_r":     True,   # deviation from mean (rounded)
        "iso_alpha": False,  # hide — already in hover_name area
        "dev":       False,  # hide raw float; show rounded version
    },
    labels = {
        "dev":       "Deviation (yrs)",
        "dev_r":     "Deviation from mean (yrs)",
        "lifeExp_r": "Life expectancy (yrs)",
    },

    # Sub-Saharan Africa drags furthest below the global average (~67 yrs),
    # with several countries 20 + years below the mean (e.g. Swaziland ≈ 39 yrs).
    title = (
        "Sub-Saharan Africa Falls Furthest Below the World Average Life Expectancy (2007)<br>"
        f"<sup>Global mean = {global_mean:.1f} yrs  |  "
        "Red = below average  ·  Blue = above average  |  "
        "Diverging scale centred at 0</sup>"
    ),
)

# ── 4. Polish ─────────────────────────────────────────────────────────────────
fig.update_layout(
    geo = dict(
        showframe      = False,
        showcoastlines = True,
        coastlinecolor = "rgba(80,80,80,0.4)",
        projection_type = "natural earth",
        bgcolor         = "rgba(0,0,0,0)",
    ),
    coloraxis_colorbar = dict(
        title      = "Deviation<br>(years)",
        tickformat = "+.0f",          # show + sign on positive ticks
        len        = 0.6,
    ),
    title_font_size = 15,
    margin = dict(l=0, r=0, t=80, b=0),
    paper_bgcolor = "#f9f9f9",
)

fig.show()

In [ ]:

# TASK 2 — Custom GeoJSON Choropleth
# Geography : India — State & Union Territory boundaries
# Variable   : Unemployment rate (%) by state, 2022–23 (CMIE / govt estimates)
# GeoJSON    : https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson


# ── 0. Install / import ──────────────────────────────────────────────────────
# (plotly is pre-installed in Colab; requests always is too)
import json, requests
import pandas as pd
import plotly.express as px


# MARKDOWN DESIGN DECISIONS

# GeoJSON source:
#   https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson
#   — A widely-used community GeoJSON with all 29 states + 7 UTs, including
#     the post-bifurcation Telangana boundary.  Freely available, no auth needed.
#
# Chart type: px.choropleth_map  (tile-backed / Mapbox-style)
#   Reason: Indian states have complex, highly irregular coastlines and
#   mountain borders.  A tile basemap (OpenStreetMap) gives geographic
#   context (rivers, cities, coastline) that makes the regional pattern
#   immediately interpretable. px.choropleth on a blank projection loses
#   that spatial anchoring for an audience unfamiliar with Indian geography.
#
# Colour scale: "YlOrRd"  → SEQUENTIAL
#   Reason: unemployment rate is a one-directional quantity (0 % → high %).
#   There is no meaningful midpoint to centre; the question is simply
#   "how much?" — more red = worse.  A diverging scale would imply a
#   natural neutral point that doesn't exist here.
#
# Insight title: Haryana & Rajasthan lead mainland unemployment;
#                Southern states cluster near the national average.



# ── 1. Fetch GeoJSON ─────────────────────────────────────────────────────────
GEOJSON_URL = (
    "https://raw.githubusercontent.com/geohacker/india/"
    "master/state/india_telengana.geojson"
)
resp = requests.get(GEOJSON_URL, timeout=30)
resp.raise_for_status()
india_geo = resp.json()

# ── 2. Inspect properties to identify featureidkey ──────────────────────────
print("── Sample feature properties ──")
print(india_geo["features"][0]["properties"])
# Expected output:  {'NAME_1': 'Andhra Pradesh', 'ID_1': 1, ...}
# ➜  featureidkey = 'properties.NAME_1'

# Collect all state names actually present in the GeoJSON
geo_names = sorted(
    f["properties"]["NAME_1"] for f in india_geo["features"]
)
print(f"\n── {len(geo_names)} regions in GeoJSON ──")
for n in geo_names:
    print(" ", n)


# ── 3. Build dataset  ────────────────────────────────────────────────────────
# Source: CMIE (Centre for Monitoring Indian Economy) state unemployment
# estimates, annual average 2022-23.  Rounded to 1 dp; a few small UTs
# are rolled into neighbouring entries where boundary data merges them.
data = {
    "state": [
        "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
        "Chhattisgarh", "Goa", "Gujarat", "Haryana",
        "Himachal Pradesh", "Jammu and Kashmir", "Jharkhand", "Karnataka",
        "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur",
        "Meghalaya", "Mizoram", "Nagaland", "Orissa",
        "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu",
        "Telangana", "Tripura", "Uttar Pradesh", "Uttarakhand",
        "West Bengal", "Delhi",
    ],
    "unemployment_rate": [
        6.4,  4.1,  7.3,  10.2,
        3.8,  9.5,  2.1,  28.4,
        6.3,  18.7,  8.9,  3.6,
        6.7,  2.9,  4.1,  7.2,
        5.8,  3.9,  6.2,  5.6,
        7.7,  24.1,  4.2,  3.4,
        5.8,  9.1,  4.2,  5.0,
        5.3,  12.8,
    ],
}
df = pd.DataFrame(data)
print(f"\n── Dataset: {len(df)} states ──")
print(df.sort_values("unemployment_rate", ascending=False).head(8))


# ── 4. Align names between GeoJSON and DataFrame ────────────────────────────
# The GeoJSON uses 'Jammu and Kashmir' — make sure df matches.
# (Already aligned above; print any mismatches for safety.)
geo_set = set(geo_names)
df_set  = set(df["state"])
mismatch = df_set - geo_set
if mismatch:
    print(f"\n⚠ Names in df not found in GeoJSON → {mismatch}")
    print("  These states will appear grey on the map.")
else:
    print("\n✓ All state names match GeoJSON — no grey polygons expected.")


# ── 5. Plot ──────────────────────────────────────────────────────────────────
fig = px.choropleth_map(
    df,
    geojson          = india_geo,
    locations        = "state",
    featureidkey     = "properties.NAME_1",  # ← identified by inspecting step 2
    color            = "unemployment_rate",
    color_continuous_scale = "YlOrRd",       # sequential: pale yellow → deep red
    range_color      = (0, 30),              # fix scale so Haryana's spike doesn't
                                             # compress all other variation
    map_style        = "carto-positron",     # clean light basemap, no token needed
    zoom             = 3.5,
    center           = {"lat": 22.5, "lon": 82.5},
    opacity          = 0.82,
    hover_name       = "state",
    hover_data       = {"unemployment_rate": ":.1f"},
    labels           = {"unemployment_rate": "Unemployment (%)"},
    title            = (
        "Haryana (28 %) & Rajasthan (24 %) Dominate India's Unemployment Map — "
        "Southern States Cluster Near the National Average (2022-23)<br>"
        "<sup>Sequential YlOrRd scale  |  "
        "featureidkey = properties.NAME_1  |  "
        "Source: CMIE estimates</sup>"
    ),
)

fig.update_layout(
    margin               = dict(l=0, r=0, t=90, b=0),
    coloraxis_colorbar   = dict(
        title      = "Unemployment<br>rate (%)",
        ticksuffix = "%",
        len        = 0.55,
    ),
    title_font_size = 14,
    paper_bgcolor   = "#f4f4f4",
)

fig.show()